In [1]:
import os
import shutil
import tarfile
from pathlib import Path

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
BASE_DIR = Path("/kaggle/working")

PREPARED_DIR = BASE_DIR / "edaic_prepared"

DEV_DIR = PREPARED_DIR / "dev"
TEST_DIR = PREPARED_DIR / "test"
LABELS_DIR = PREPARED_DIR / "labels"

DEV_PARTICIPANTS_DIR = DEV_DIR / "participants"
TEST_PARTICIPANTS_DIR = TEST_DIR / "participants"

TEMP_DIR = BASE_DIR / "edaic_tmp"

for directory in [
    DEV_PARTICIPANTS_DIR,
    TEST_PARTICIPANTS_DIR,
    LABELS_DIR,
    TEMP_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Prepared directory:")
print(PREPARED_DIR)

Prepared directory:
/kaggle/working/edaic_prepared


In [3]:
DATASET_URL = (
    "https://dcapswoz.ict.usc.edu/wwwedaic"
)

label_urls = {
    "dev_split.csv":
        f"{DATASET_URL}/labels/dev_split.csv",

    "test_split.csv":
        f"{DATASET_URL}/labels/test_split.csv",

    "Detailed_PHQ8_Labels.csv":
        f"{DATASET_URL}/labels/Detailed_PHQ8_Labels.csv",

    "detailed_lables.csv":
        f"{DATASET_URL}/labels/detailed_lables.csv"
}

for filename, url in label_urls.items():

    output_path = (
        LABELS_DIR / filename
    )

    if output_path.exists():

        print(
            f"{filename} already exists."
        )

        continue

    print(
        f"Downloading {filename}..."
    )

    command = (
        f"wget -q "
        f"-O '{output_path}' "
        f"'{url}'"
    )

    result = os.system(command)

    if result != 0:

        raise RuntimeError(
            f"Failed to download {filename}"
        )

    print(
        f"{filename} downloaded."
    )

print("\nAll label files downloaded.")

dev_split.csv downloaded.
test_split.csv downloaded.
Detailed_PHQ8_Labels.csv downloaded.
detailed_lables.csv downloaded.

All label files downloaded.


In [4]:
dev = pd.read_csv(
    LABELS_DIR / "dev_split.csv"
)

test = pd.read_csv(
    LABELS_DIR / "test_split.csv"
)

phq_details = pd.read_csv(
    LABELS_DIR / "Detailed_PHQ8_Labels.csv"
)

detailed_labels = pd.read_csv(
    LABELS_DIR / "detailed_lables.csv"
)

print("Dev:", dev.shape)
print("Test:", test.shape)
print("Detailed PHQ:", phq_details.shape)
print("Detailed labels:", detailed_labels.shape)

Dev: (56, 6)
Test: (56, 6)
Detailed PHQ: (219, 10)
Detailed labels: (275, 33)


In [5]:
def aggregate_features(df, prefix):

    feature_cols = [
        c for c in df.columns
        if c not in [
            "name",
            "frameTime"
        ]
    ]

    x = df[feature_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    # OpenSMILE invalid/sentinel value
    x = x.replace(
        -201,
        np.nan
    )

    # Infinite values
    x = x.replace(
        [np.inf, -np.inf],
        np.nan
    )

    result = {}

    for col in feature_cols:

        result[
            f"{prefix}_{col}_mean"
        ] = x[col].mean()

        result[
            f"{prefix}_{col}_std"
        ] = x[col].std()

        result[
            f"{prefix}_{col}_median"
        ] = x[col].median()

    return pd.Series(result)

In [6]:
DATA_BASE_URL = (
    "https://dcapswoz.ict.usc.edu/wwwedaic/data"
)


def download_and_extract_participant(
    pid,
    split_dir
):

    pid = int(pid)

    participant_dir = (
        split_dir /
        "participants" /
        f"{pid}_P"
    )

    features_dir = (
        participant_dir /
        "features"
    )

    participant_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    features_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Files we want to retain
    required_files = {

        f"{pid}_Transcript.csv":
            participant_dir /
            f"{pid}_Transcript.csv",

        f"{pid}_OpenSMILE2.3.0_egemaps.csv":
            features_dir /
            f"{pid}_OpenSMILE2.3.0_egemaps.csv",

        f"{pid}_OpenSMILE2.3.0_mfcc.csv":
            features_dir /
            f"{pid}_OpenSMILE2.3.0_mfcc.csv"
    }

    # If all files already exist, skip download
    if all(
        path.exists()
        for path in required_files.values()
    ):

        print(
            f"{pid}: files already exist."
        )

        return (
            required_files[
                f"{pid}_OpenSMILE2.3.0_egemaps.csv"
            ],
            required_files[
                f"{pid}_OpenSMILE2.3.0_mfcc.csv"
            ]
        )

    archive = (
        TEMP_DIR /
        f"{pid}_P.tar.gz"
    )

    url = (
        f"{DATA_BASE_URL}/{pid}_P.tar.gz"
    )

    print(
        f"{pid}: downloading..."
    )

    # Silent wget
    command = (
        f"wget -q "
        f"-O '{archive}' "
        f"'{url}'"
    )

    result = os.system(command)

    if (
        result != 0
        or not archive.exists()
    ):

        raise RuntimeError(
            f"Download failed for participant {pid}"
        )

    print(
        f"{pid}: extracting required files..."
    )

    try:

        with tarfile.open(
            archive,
            "r:gz"
        ) as tar:

            found = set()

            for member in tar:

                basename = os.path.basename(
                    member.name
                )

                if basename not in required_files:
                    continue

                output_path = (
                    required_files[
                        basename
                    ]
                )

                file_obj = tar.extractfile(
                    member
                )

                if file_obj is None:
                    continue

                with open(
                    output_path,
                    "wb"
                ) as output:

                    while True:

                        chunk = file_obj.read(
                            1024 * 1024
                        )

                        if not chunk:
                            break

                        output.write(chunk)

                found.add(basename)

        missing = (
            set(required_files.keys())
            - found
        )

        if missing:

            raise FileNotFoundError(
                f"{pid}: missing files: {missing}"
            )

        print(
            f"{pid}: complete."
        )

        return (
            required_files[
                f"{pid}_OpenSMILE2.3.0_egemaps.csv"
            ],
            required_files[
                f"{pid}_OpenSMILE2.3.0_mfcc.csv"
            ]
        )

    finally:

        # Delete large archive
        if archive.exists():
            archive.unlink()

In [7]:
def process_participant(
    pid,
    split_name,
    split_dir,
    labels
):

    pid = int(pid)

    # Download and save files
    eg_path, mfcc_path = (
        download_and_extract_participant(
            pid,
            split_dir
        )
    )

    # Load acoustic files
    eg = pd.read_csv(
        eg_path,
        sep=";"
    )

    mfcc = pd.read_csv(
        mfcc_path,
        sep=";"
    )

    # Aggregate acoustic features
    eg_agg = aggregate_features(
        eg,
        "eg"
    )

    mfcc_agg = aggregate_features(
        mfcc,
        "mfcc"
    )

    # Combine features
    row = pd.concat(
        [
            eg_agg,
            mfcc_agg
        ]
    )

    # Store basic temporal metadata
    row[
        "Feature_Duration_sec"
    ] = eg["frameTime"].max()

    row[
        "eGeMAPS_Frames"
    ] = len(eg)

    row[
        "MFCC_Frames"
    ] = len(mfcc)

    row[
        "Participant_ID"
    ] = pid

    row = row.to_frame().T

    # Add participant labels
    row = row.merge(
        labels[
            [
                "Participant_ID",
                "Gender",
                "PHQ_Binary",
                "PHQ_Score",
                "PCL-C (PTSD)",
                "PTSD Severity"
            ]
        ],
        on="Participant_ID",
        how="left"
    )

    row["Split"] = split_name

    return row

In [8]:
DEV_FEATURE_FILE = (
    DEV_DIR /
    "edaic_audio_features_dev.csv"
)

dev_ids = (
    dev["Participant_ID"]
    .astype(int)
    .tolist()
)

processed_dev = set()

if DEV_FEATURE_FILE.exists():

    old_dev = pd.read_csv(
        DEV_FEATURE_FILE
    )

    processed_dev = set(
        old_dev[
            "Participant_ID"
        ].astype(int)
    )

    print(
        f"Already processed Dev participants: "
        f"{len(processed_dev)}"
    )

failed_dev = []

for i, pid in enumerate(dev_ids, start=1):

    if pid in processed_dev:

        print(
            f"[{i}/{len(dev_ids)}] "
            f"{pid} already processed"
        )

        continue

    try:

        print(
            f"\n[{i}/{len(dev_ids)}] "
            f"Processing Dev participant {pid}"
        )

        row = process_participant(
            pid,
            "dev",
            DEV_DIR,
            dev
        )

        row.to_csv(
            DEV_FEATURE_FILE,
            mode="a",
            header=(
                not DEV_FEATURE_FILE.exists()
            ),
            index=False
        )

        processed_dev.add(pid)

        print(
            f"✓ {pid} saved"
        )

    except Exception as e:

        print(
            f"✗ {pid} FAILED: {e}"
        )

        failed_dev.append(pid)


print("\nDEV COMPLETE")
print(
    "Successful:",
    len(processed_dev)
)
print(
    "Failed:",
    len(failed_dev)
)

if failed_dev:
    print(
        "Failed IDs:",
        failed_dev
    )


[1/56] Processing Dev participant 300
300: downloading...
300: extracting required files...
300: complete.
✓ 300 saved

[2/56] Processing Dev participant 301
301: downloading...
301: extracting required files...
301: complete.
✓ 301 saved

[3/56] Processing Dev participant 306
306: downloading...
306: extracting required files...
306: complete.
✓ 306 saved

[4/56] Processing Dev participant 317
317: downloading...
317: extracting required files...
317: complete.
✓ 317 saved

[5/56] Processing Dev participant 320
320: downloading...
320: extracting required files...
320: complete.
✓ 320 saved

[6/56] Processing Dev participant 321
321: downloading...
321: extracting required files...
321: complete.
✓ 321 saved

[7/56] Processing Dev participant 331
331: downloading...
331: extracting required files...
331: complete.
✓ 331 saved

[8/56] Processing Dev participant 334
334: downloading...
334: extracting required files...
334: complete.
✓ 334 saved

[9/56] Processing Dev participant 336
3

In [9]:
TEST_FEATURE_FILE = (
    TEST_DIR /
    "edaic_audio_features_test.csv"
)

test_ids = (
    test["Participant_ID"]
    .astype(int)
    .tolist()
)

processed_test = set()

if TEST_FEATURE_FILE.exists():

    old_test = pd.read_csv(
        TEST_FEATURE_FILE
    )

    processed_test = set(
        old_test[
            "Participant_ID"
        ].astype(int)
    )

    print(
        f"Already processed Test participants: "
        f"{len(processed_test)}"
    )

failed_test = []

for i, pid in enumerate(
    test_ids,
    start=1
):

    if pid in processed_test:

        print(
            f"[{i}/{len(test_ids)}] "
            f"{pid} already processed"
        )

        continue

    try:

        print(
            f"\n[{i}/{len(test_ids)}] "
            f"Processing Test participant {pid}"
        )

        row = process_participant(
            pid,
            "test",
            TEST_DIR,
            test
        )

        row.to_csv(
            TEST_FEATURE_FILE,
            mode="a",
            header=(
                not TEST_FEATURE_FILE.exists()
            ),
            index=False
        )

        processed_test.add(pid)

        print(
            f"✓ {pid} saved"
        )

    except Exception as e:

        print(
            f"✗ {pid} FAILED: {e}"
        )

        failed_test.append(pid)


print("\nTEST COMPLETE")
print(
    "Successful:",
    len(processed_test)
)
print(
    "Failed:",
    len(failed_test)
)

if failed_test:
    print(
        "Failed IDs:",
        failed_test
    )


[1/56] Processing Test participant 600
600: downloading...
600: extracting required files...
600: complete.
✓ 600 saved

[2/56] Processing Test participant 602
602: downloading...
602: extracting required files...
602: complete.
✓ 602 saved

[3/56] Processing Test participant 604
604: downloading...
604: extracting required files...
604: complete.
✓ 604 saved

[4/56] Processing Test participant 605
605: downloading...
605: extracting required files...
605: complete.
✓ 605 saved

[5/56] Processing Test participant 606
606: downloading...
606: extracting required files...
606: complete.
✓ 606 saved

[6/56] Processing Test participant 607
607: downloading...
607: extracting required files...
607: complete.
✓ 607 saved

[7/56] Processing Test participant 609
609: downloading...
609: extracting required files...
609: complete.
✓ 609 saved

[8/56] Processing Test participant 615
615: downloading...
615: extracting required files...
615: complete.
✓ 615 saved

[9/56] Processing Test particip

In [10]:
dev_features = pd.read_csv(
    DEV_FEATURE_FILE
)

test_features = pd.read_csv(
    TEST_FEATURE_FILE
)

print(
    "DEV:",
    dev_features.shape
)

print(
    "TEST:",
    test_features.shape
)

DEV: (56, 196)
TEST: (56, 196)


In [11]:
print(
    "Dev participants:",
    dev_features[
        "Participant_ID"
    ].nunique()
)

print(
    "Test participants:",
    test_features[
        "Participant_ID"
    ].nunique()
)

Dev participants: 56
Test participants: 56


In [12]:
expected_dev = set(
    dev["Participant_ID"]
    .astype(int)
)

expected_test = set(
    test["Participant_ID"]
    .astype(int)
)

actual_dev = set(
    dev_features[
        "Participant_ID"
    ].astype(int)
)

actual_test = set(
    test_features[
        "Participant_ID"
    ].astype(int)
)

print(
    "Missing Dev:",
    sorted(
        expected_dev - actual_dev
    )
)

print(
    "Missing Test:",
    sorted(
        expected_test - actual_test
    )
)

Missing Dev: []
Missing Test: []


In [13]:
print("\nDEV participant count:")
print(
    len(
        list(
            DEV_PARTICIPANTS_DIR.iterdir()
        )
    )
)

print("\nTEST participant count:")
print(
    len(
        list(
            TEST_PARTICIPANTS_DIR.iterdir()
        )
    )
)


DEV participant count:
56

TEST participant count:
56


In [14]:

print("\nDEV example:")
!find /kaggle/working/edaic_prepared/dev/participants -maxdepth 2 -type f | head -6

print("\nTEST example:")
!find /kaggle/working/edaic_prepared/test/participants -maxdepth 2 -type f | head -6



DEV example:
/kaggle/working/edaic_prepared/dev/participants/423_P/423_Transcript.csv
/kaggle/working/edaic_prepared/dev/participants/465_P/465_Transcript.csv
/kaggle/working/edaic_prepared/dev/participants/454_P/454_Transcript.csv
/kaggle/working/edaic_prepared/dev/participants/301_P/301_Transcript.csv
/kaggle/working/edaic_prepared/dev/participants/479_P/479_Transcript.csv
/kaggle/working/edaic_prepared/dev/participants/442_P/442_Transcript.csv

TEST example:
/kaggle/working/edaic_prepared/test/participants/618_P/618_Transcript.csv
/kaggle/working/edaic_prepared/test/participants/664_P/664_Transcript.csv
/kaggle/working/edaic_prepared/test/participants/715_P/715_Transcript.csv
/kaggle/working/edaic_prepared/test/participants/620_P/620_Transcript.csv
/kaggle/working/edaic_prepared/test/participants/634_P/634_Transcript.csv
/kaggle/working/edaic_prepared/test/participants/651_P/651_Transcript.csv
